# Q1 Validation Notebook

This notebook validates the calculations and chart logic currently used in the Streamlit `Product Analysis` page.

It reproduces:
- Research API acquisition and churn KPI
- Retention comparison (first action = query vs research)
- Pareto concentration of research traffic
- Average response time by model (`mini` vs `pro`)


In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
import plotly.express as px

APP_DIR = Path.cwd()
ZIP_PATH = APP_DIR / "data.zip"

required_files = [
    "hourly_usage.csv",
    "infrastructure_costs.csv",
    "research_requests.csv",
    "users.csv",
]

frames = {}
if ZIP_PATH.is_file():
    try:
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            members = set(zf.namelist())
            missing = [name for name in required_files if name not in members]
            if not missing:
                for name in required_files:
                    with zf.open(name) as f:
                        frames[name] = pd.read_csv(f)
    except zipfile.BadZipFile:
        frames = {}

if not frames:
    for name in required_files:
        for candidate in (APP_DIR / name, APP_DIR.parent / name):
            if candidate.is_file():
                frames[name] = pd.read_csv(candidate)
                break
        else:
            raise FileNotFoundError(f"Could not load '{name}' from zip or CSV fallback.")

hourly_usage = frames["hourly_usage.csv"].copy()
Infrastructure_costs = frames["infrastructure_costs.csv"].copy()
research_requests = frames["research_requests.csv"].copy()
users = frames["users.csv"].copy()

for df in (hourly_usage, Infrastructure_costs, research_requests, users):
    df.columns = df.columns.str.lower()

print("Loaded:")
print("hourly_usage", hourly_usage.shape)
print("Infrastructure_costs", Infrastructure_costs.shape)
print("research_requests", research_requests.shape)
print("users", users.shape)


In [ ]:
def build_events(hourly_df: pd.DataFrame, research_df: pd.DataFrame) -> pd.DataFrame:
    h = hourly_df[["user_id", "hour"]].copy()
    h["event_ts"] = pd.to_datetime(h["hour"], errors="coerce", utc=True)
    h["source"] = "query"

    r = research_df[["user_id", "timestamp"]].copy()
    r["event_ts"] = pd.to_datetime(r["timestamp"], errors="coerce", utc=True)
    r["source"] = "research"

    events = pd.concat([
        h[["user_id", "event_ts", "source"]],
        r[["user_id", "event_ts", "source"]],
    ], ignore_index=True)

    events["user_id"] = pd.to_numeric(events["user_id"], errors="coerce")
    events = events.dropna(subset=["user_id", "event_ts"]).copy()
    events["user_id"] = events["user_id"].astype(int)
    return events.sort_values(["user_id", "event_ts", "source"]).reset_index(drop=True)


events = build_events(hourly_usage, research_requests)

users_clean = users[["user_id", "created_at"]].copy()
users_clean["created_at"] = pd.to_datetime(users_clean["created_at"], errors="coerce", utc=True)
users_clean["user_id"] = pd.to_numeric(users_clean["user_id"], errors="coerce")
users_clean = users_clean.dropna(subset=["user_id", "created_at"]).copy()
users_clean["user_id"] = users_clean["user_id"].astype(int)

# Keep only valid activity on/after account creation.
events = events.merge(users_clean, on="user_id", how="inner")
events = events.loc[events["event_ts"] >= events["created_at"], ["user_id", "event_ts", "source"]]
events = events.sort_values(["user_id", "event_ts", "source"]).reset_index(drop=True)

first_actions = (
    events.groupby("user_id", as_index=False)
    .first()[["user_id", "event_ts", "source"]]
    .rename(columns={"event_ts": "first_event_ts", "source": "first_source"})
)
last_activity = (
    events.groupby("user_id", as_index=False)["event_ts"]
    .max()
    .rename(columns={"event_ts": "last_event_ts"})
)
lifecycle = first_actions.merge(last_activity, on="user_id", how="inner")
lifecycle["retained_30d"] = lifecycle["last_event_ts"] >= lifecycle["first_event_ts"] + pd.Timedelta(days=30)

lifecycle.head()


In [ ]:
# KPI validation: acquisition + churn
rr_launch = research_requests.copy()
rr_launch["timestamp"] = pd.to_datetime(rr_launch["timestamp"], errors="coerce", utc=True)
launch_ts = rr_launch["timestamp"].dropna().min()

new_users = users_clean.loc[users_clean["created_at"] > launch_ts, ["user_id"]].drop_duplicates()
new_lifecycle = new_users.merge(lifecycle, on="user_id", how="inner")

if len(new_lifecycle) == 0:
    acquisition_pct = 0.0
    churn_pct = 0.0
else:
    research_first = new_lifecycle["first_source"].eq("research")
    acquisition_pct = 100.0 * research_first.mean()
    research_first_users = new_lifecycle.loc[research_first]
    if len(research_first_users) == 0:
        churn_pct = 0.0
    else:
        retention_pct = 100.0 * research_first_users["retained_30d"].mean()
        churn_pct = 100.0 - retention_pct

print(f"Launch timestamp: {launch_ts}")
print(f"New users after launch: {len(new_users):,}")
print(f"Research API Acquisition: {acquisition_pct:.2f}%")
print(f"Churn among Research-first users: {churn_pct:.2f}%")


In [ ]:
# Chart prototype 1: Retention comparison
retention_df = lifecycle.copy()
retention_df["segment"] = retention_df["first_source"].map({
    "query": "First Action = Query",
    "research": "First Action = Research",
})
retention_df = (
    retention_df.groupby("segment", as_index=False)["retained_30d"]
    .mean()
    .rename(columns={"retained_30d": "retention_rate"})
)
retention_df["retention_rate"] = retention_df["retention_rate"] * 100.0

fig_retention = px.bar(
    retention_df,
    x="segment",
    y="retention_rate",
    title="Retention Rate by First Action",
    labels={"segment": "First Action", "retention_rate": "Retention Rate (%)"},
    text=retention_df["retention_rate"].map(lambda x: f"{x:.1f}%"),
)
fig_retention.update_layout(template="simple_white")
fig_retention.show()


In [ ]:
# Chart prototype 2: Average response time by model (mini vs pro)
latency_df = research_requests.copy()
latency_df["model"] = latency_df["model"].astype(str).str.lower().str.strip()
latency_df["response_time_seconds"] = pd.to_numeric(latency_df["response_time_seconds"], errors="coerce")
latency_df = (
    latency_df[latency_df["model"].isin(["mini", "pro"])]
    .dropna(subset=["response_time_seconds"])
    .groupby("model", as_index=False)["response_time_seconds"]
    .mean()
    .rename(columns={"response_time_seconds": "avg_response_time_seconds"})
)

fig_latency = px.bar(
    latency_df,
    x="avg_response_time_seconds",
    y="model",
    orientation="h",
    title="Average Response Time by Model (Mini vs Pro)",
    labels={
        "avg_response_time_seconds": "Average Response Time (seconds)",
        "model": "Model",
    },
    text=latency_df["avg_response_time_seconds"].map(lambda x: f"{x:.2f}s"),
)
fig_latency.update_layout(template="simple_white")
fig_latency.show()


In [ ]:
# Chart prototype 3: Pareto concentration of research traffic
pareto_raw = research_requests.copy()
pareto_raw["user_id"] = pd.to_numeric(pareto_raw["user_id"], errors="coerce")
pareto_raw = pareto_raw.dropna(subset=["user_id"]).copy()
pareto_raw["user_id"] = pareto_raw["user_id"].astype(int)

counts = pareto_raw.groupby("user_id").size().sort_values(ascending=False)
pareto = pd.DataFrame({"requests": counts.values})
pareto["cum_requests_pct"] = 100.0 * pareto["requests"].cumsum() / pareto["requests"].sum()
pareto["cum_users_pct"] = 100.0 * (pareto.index + 1) / len(pareto)
pareto = pd.concat(
    [
        pd.DataFrame({"cum_users_pct": [0.0], "cum_requests_pct": [0.0]}),
        pareto[["cum_users_pct", "cum_requests_pct"]],
    ],
    ignore_index=True,
)

fig_pareto = px.line(
    pareto,
    x="cum_users_pct",
    y="cum_requests_pct",
    title="Research API Traffic Concentration (Pareto Curve)",
    labels={
        "cum_users_pct": "Cumulative % of Users",
        "cum_requests_pct": "Cumulative % of Total Requests",
    },
)
y_at_5 = float(
    pareto.loc[pareto["cum_users_pct"] >= 5.0, "cum_requests_pct"].head(1).fillna(0.0).iloc[0]
)
fig_pareto.add_vline(x=5.0, line_dash="dash", line_color="gray")
fig_pareto.add_hline(y=y_at_5, line_dash="dash", line_color="gray")
fig_pareto.update_layout(template="simple_white")
fig_pareto.show()

print(f"At 5% of users, cumulative request share is {y_at_5:.2f}%")
